In [ ]:
# IROS optimisation by hand - SCOX-1

from singleCAM_IROS._pipeline_support import _handle_dirpaths

import numpy as np
from numpy.typing import NDArray
import pandas as pd

from bloodmoon.types import CoordEquatorial
from bloodmoon.coords import shift2pos, shift2angle, angle2shift, equatorial2shift
from bloodmoon.io import simulation_files
from bloodmoon.mask import CodedMaskCamera, codedmask, count, decode
from bloodmoon.optim import model_sky as bm_sky

import darksun as ds
from darksun.data import DataLoader, CatalogueLoader

from fract_shift2 import model_shadowgram


def extract_catalogue_angular_coords(
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
) -> pd.DataFrame:
    """
    """
    s, txs, tys = [], [], []
    for sourceID in np.unique(catalogue.DLdata['ID']):
        source = catalogue.DLdata[(catalogue.DLdata['ID'] == sourceID)]
        sx, sy = equatorial2shift(sdl, camera, source['RA'][0], source['DEC'][0])
        #sx, sy = equatorial2shift(sdl, camera, source['RA'], source['DEC'])
        thetax, thetay = map(lambda x: shift2angle(wfm, x), (sx, sy))
        s.append(sourceID)
        txs.append(thetax)
        tys.append(thetay)

    df = pd.DataFrame(
        {'ID': s, 'THETAX': txs, 'THETAY': tys}
    )
    return df

def model_sky(
    camera: CodedMaskCamera,
    shift_x: float,
    shift_y: float,
    fluence: float,
    vignetting: bool = True,
    psfy: bool = True,
    bkg: NDArray | None = None,
) -> NDArray:
    """
    Generate a model of the reconstructed sky image for a point source.
    """
    detector = model_shadowgram(
        camera, shift_x, shift_y, vignetting=vignetting, psfy=psfy,
    ) * fluence
    if bkg is not None:
        detector += bkg
    return decode(camera, detector)


#skyfield = "IROSDummy"
#mask_FITS = "wfm_mask_NTHT_20250725.fits"
##mask_FITS = "wfm_mask_summer2021.fits"
#data_FITS = "scox1_2-50keV_mask_050_1040x17_1ks_infdet_opaquemask_newpathinmask"

skyfield = "GalacticCentre"
mask_FITS = "wfm_mask_NTHT_20250725.fits"
data_FITS = "galctr_rxte-sax_2-50keV_mask_050_1040x17_opaquemask_infdet"

mask_path, simul_data, _ = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
)

VIGNETTING = True
PSFY = False
UPX, UPY = 30, 4
wfm = codedmask(mask_path, UPX, UPY)

cam_a = "cam1a"
cam_b = "cam1b"
dataset = 'detected'

filepaths = simulation_files(simul_data)

source_ID = 'scox1'

#from var import detector_solid_angle
#
#rng = np.random.default_rng()
#
#omega = detector_solid_angle(wfm)
#cxb_detector = omega / omega.sum() * rng.poisson(17400, (wfm.shape_detector)).sum() * wfm.bulk


from pathlib import Path

def savefig_to(
    figpath: str | Path,
    name: str,
    frmt: str = 'png',
) -> str | Path | None:
    """
    Creates the filepath to save a Figure with the chosen format,
    and checks if the file already exists. If the file exists,
    `None` is returned to avoid the file overwriting. 
    """
    def is_img(filepath: Path) -> bool:
        """Checks if a file has been already saved."""
        return filepath.is_file()
    
    filepath = f'{figpath}/{name}.{frmt}'
    if not is_img(Path(filepath)):
        return filepath
    return None

save_to = '/home/edoardo/Desktop/PSF_test'

In [ ]:
#from bloodmoon.optim import _detector_footprint_cached
#from darksun.images import upscale, downscale
#from fract_shift2 import fshift, apply_vignetting, apply_detector_resolution
#
#def model_shadowgram(
#    camera: CodedMaskCamera,
#    shift_x: float,
#    shift_y: float,
#    vignetting: bool = True,
#    psfy: bool = True,
#) -> NDArray:
#    """
#    Generates a normalized shadowgram for a point source
#    with fractional shift of the mask pattern.
#    """
#    FY, FX = 4, 20
#    upsampled_mask = upscale(
#        data=camera.mask.astype(float), 
#        upscale_y=FY,
#        upscale_x=FX,
#    ) * np.prod((FY, FX))
#    
#    # instrumental effects and shift mask pattern
#    # - apply vignetting to mask pattern array
#    mask_vignetted = (
#        apply_vignetting(camera, upsampled_mask, shift_x, shift_y)
#        if vignetting else upsampled_mask
#    )
#    # - shift mask array to match source direction
#    pxdimy, pxdimx = (
#        camera.specs['mask_deltay'] / camera.upscale_f.y,
#        camera.specs['mask_deltax'] / camera.upscale_f.x,
#    )
#    fr, fc = (
#        (-1.0) * shift_y * FY / pxdimy,
#        (-1.0) * shift_x * FX / pxdimx,
#    )
#    mask_shifted = fshift(mask_vignetted, fr, fc)
#    # - apply detector spatial resolution
#    sg = (
#        apply_detector_resolution(camera, mask_shifted)
#        if psfy else mask_shifted
#    )
#
#    downsampled_sg = downscale(
#        data=sg,
#        downscale_y=FY,
#        downscale_x=FX,
#    ) / np.prod((FY, FX))
#
#    # extract normalised detector image
#    i_min, i_max, j_min, j_max = _detector_footprint_cached(camera)
#    detector = downsampled_sg[i_min:i_max, j_min:j_max]
#    detector *= camera.bulk
#    detector /= np.sum(detector)
#    print(
#        f'Original mask shape: {camera.shape_mask}\n'
#        f'Downsampled mask shape: {downsampled_sg.shape}\n'
#        f'Original detector shape: {camera.shape_detector}\n'
#        f'Downsampled detector shape: {detector.shape}\n'
#    )
#    
#    return detector
#
#
#def model_sky(
#    camera: CodedMaskCamera,
#    shift_x: float,
#    shift_y: float,
#    fluence: float,
#    vignetting: bool = True,
#    psfy: bool = True,
#) -> NDArray:
#    """
#    Generate a model of the reconstructed sky image for a point source.
#    """
#    detector = model_shadowgram(
#        camera, shift_x, shift_y, vignetting=vignetting, psfy=psfy,
#    ) * fluence
#    return decode(camera, detector)

In [ ]:
from astropy.io.fits.fitsrec import FITS_rec

def select_source_photons(
    coords: CoordEquatorial | tuple[CoordEquatorial, ...],
    data: FITS_rec,
) -> FITS_rec:
    """
    Selects photon events relative to the input source RA/Dec coords.
    """
    mask = np.ones(len(data), dtype=bool)
    for c in ((coords,) if isinstance(coords, CoordEquatorial) else coords):
        mask &= (
            (np.isclose(data['RA'], c.ra) & np.isclose(data['DEC'], c.dec))
        )
    selected = data[mask]
    print(f'Selected {len(selected)} ph out of {len(data)} ph.')
    return selected


def collapse_view(arr: NDArray) -> tuple[NDArray, NDArray]:
    """
    Collapses input 2D array by adding the elements along
    one axis, for both the (x, y) array axes.

    Returns:
        output (tuple[NDArray, NDArray]):
            - collapsed array along the y axis
            - collapsed array along the x axis
    
    ## Notes:
        - if `arr.shape = (n, m)`, the collapsed array
          along y has length `m`, while the collapsed
          array along x has length `n`.
    """
    return np.sum(arr, axis=0), np.sum(arr, axis=1)


def extract_section(
    arr: NDArray,
    rows: slice,
    cols: slice,
) -> NDArray:
    """Returns an array view of the input array."""
    return arr[rows, cols]


sdlA = ds.get_data(filepaths[cam_a][dataset])
catalogueA = ds.get_catalogue(filepaths[cam_a]['sources'])

source_rec = catalogueA.DLdata[(catalogueA.DLdata['ID'] == source_ID)]
coords = CoordEquatorial(source_rec['RA'][0], source_rec['DEC'][0])
source_photons = select_source_photons(coords=coords, data=sdlA.DLdata)

shiftx, shifty = equatorial2shift(sdlA, wfm, coords.ra, coords.dec)
thetax, thetay = map(lambda x: shift2angle(wfm, x), (shiftx, shifty))
print(f'\n{source_ID.upper()} angular coords: {thetax, thetay}\n')

detector, _ = count(wfm, source_photons)
true_sky = decode(wfm, detector)
i, j = shift2pos(wfm, shiftx, shifty)
fluence = 1.0 * np.max(true_sky[i - 5 * UPY : i + 5 * UPY, j - 2 * UPX : j + 2 * UPX])

source_sg = model_shadowgram(wfm, shiftx, shifty, VIGNETTING, PSFY) * fluence

In [ ]:
slicerows, slicecols = slice(675, 795), slice(7200, 7750)
#slicerows, slicecols = slice(170, 200), slice(1165, 1290)
ext_det, ext_sg = map(
    lambda x: extract_section(x, slicerows, slicecols),
    (detector, source_sg),
)
det_x, det_y = collapse_view(ext_det)
sg_x, sg_y = collapse_view(ext_sg) 

ASPECT = ext_det.shape[1] / ext_det.shape[0] # (wfm.specs['mask_deltay'] / UPY) / (wfm.specs['mask_deltax'] / UPX)
dmap_img_det = ds.map4image(
    img=ext_det,
    title='Detector',
    ylabel='counts [ph]',
    xlabel='pixel index',
    img_kwargs={
        'vmin': 0,
        'vmax': ext_sg.max(),
        'aspect': ASPECT,
    }
)
dmap_img_sg = ds.map4image(
    img=ext_sg,
    title='Modelled SG',
    ylabel='counts [ph]',
    xlabel='pixel index',
    img_kwargs={
        'vmin': 0,
        'aspect': ASPECT,
    }
)
ds.image_plot((dmap_img_det, dmap_img_sg), ncols=2)

dmap_collapsed_x = ds.map4plot(
    arrs=(det_x, sg_x),
    title='X direction',
    ylabel='counts [ph]',
    xlabel='pixel index',
    labels=('detector', 'modelled'),
    x=map(
        lambda x: np.arange(len(x) + 1) - 0.5,
        (det_x, sg_x)
    ),
    color=('DodgerBlue', 'OrangeRed'),
    style='stairs',
)
dmap_collapsed_y = ds.map4plot(
    arrs=(det_y, sg_y),
    title='Y direction',
    ylabel='counts [ph]',
    xlabel='pixel index',
    labels=('detector', 'modelled'),
    x=map(
        lambda x: np.arange(len(x) + 1) - 0.5,
        (det_y, sg_y)
    ),
    color=('DodgerBlue', 'OrangeRed'),
    style='stairs',
)
ds.plot((dmap_collapsed_x, dmap_collapsed_y), ncols=2)

In [ ]:
sdlB = ds.get_data(filepaths[cam_b][dataset])
catalogueB = ds.get_catalogue(filepaths[cam_b]['sources'])

source_rec = catalogueB.DLdata[(catalogueB.DLdata['ID'] == source_ID)]
coords = CoordEquatorial(source_rec['RA'][0], source_rec['DEC'][0])
source_photons = select_source_photons(coords=coords, data=sdlB.DLdata)

shiftx, shifty = equatorial2shift(sdlB, wfm, coords.ra, coords.dec)
thetax, thetay = map(lambda x: shift2angle(wfm, x), (shiftx, shifty))
print(f'\n{source_ID.upper()} angular coords: {thetax, thetay}\n')

detector, _ = count(wfm, source_photons)
true_sky = decode(wfm, detector)
i, j = shift2pos(wfm, shiftx, shifty)
fluence = 1.0 * np.max(true_sky[i - 5 * UPY : i + 5 * UPY, j - 2 * UPX : j + 2 * UPX])

source_sg = model_shadowgram(wfm, shiftx, shifty, VIGNETTING, PSFY) * fluence

In [ ]:
slicerows, slicecols = slice(166, 195), slice(1100, 1185)
ext_det, ext_sg = map(
    lambda x: extract_section(x, slicerows, slicecols),
    (detector, source_sg),
)
det_x, det_y = collapse_view(ext_det)
sg_x, sg_y = collapse_view(ext_sg) 

ASPECT = ext_det.shape[1] / ext_det.shape[0] # (wfm.specs['mask_deltay'] / UPY) / (wfm.specs['mask_deltax'] / UPX)
dmap_img_det = ds.map4image(
    img=ext_det,
    title='Detector',
    ylabel='counts [ph]',
    xlabel='pixel index',
    img_kwargs={
        'vmin': 0,
        'vmax': ext_sg.max(),
        'aspect': ASPECT,
    }
)
dmap_img_sg = ds.map4image(
    img=ext_sg,
    title='Modelled SG',
    ylabel='counts [ph]',
    xlabel='pixel index',
    img_kwargs={
        'vmin': 0,
        'aspect': ASPECT,
    }
)
ds.image_plot((dmap_img_det, dmap_img_sg), ncols=2)

dmap_collapsed_x = ds.map4plot(
    arrs=(det_x, sg_x),
    title='X direction',
    ylabel='counts [ph]',
    xlabel='pixel index',
    labels=('detector', 'modelled'),
    x=map(
        lambda x: np.arange(len(x) + 1) - 0.5,
        (det_x, sg_x)
    ),
    color=('DodgerBlue', 'OrangeRed'),
    style='stairs',
)
dmap_collapsed_y = ds.map4plot(
    arrs=(det_y, sg_y),
    title='Y direction',
    ylabel='counts [ph]',
    xlabel='pixel index',
    labels=('detector', 'modelled'),
    x=map(
        lambda x: np.arange(len(x) + 1) - 0.5,
        (det_y, sg_y)
    ),
    color=('DodgerBlue', 'OrangeRed'),
    style='stairs',
)
ds.plot((dmap_collapsed_x, dmap_collapsed_y), ncols=2)

In [6]:
import numpy as np

from bloodmoon.coords import shift2angle, angle2shift
from bloodmoon.mask import codedmask

from fract_shift2 import fshift

mask_path = '/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/wfm_mask_NTHT_20250725.fits'
wfm = codedmask(mask_path)

In [16]:
mask = np.array(
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    dtype=float,
)

#theta_x, theta_y = 20.0, 20.0     # [deg]

pxdim_x, pxdim_y = 1.0, 1.0       # [mm]
shift_x, shift_y = -1.2, -1.2   # [mm]

theta_x, theta_y = map(lambda x: shift2angle(wfm, x), (shift_x, shift_y))   # [deg]
px_shift_x, px_shift_y = (
    (-1) * shift_x / pxdim_x,
    (-1) * shift_y / pxdim_y,
)

erosion_x, erosion_y = map(
    lambda angle, pxdim: wfm.specs['mask_thickness'] * np.tan(np.deg2rad(angle)) / pxdim,
    (theta_x, theta_y),
    (pxdim_x, pxdim_y),
)

_, fract = divmod(abs(px_shift_x), 1.0)
cut_x = erosion_x - np.sign(erosion_x) * fract

shifted_x = fshift(mask, 0, shift_x)
anti_shifted_x = fshift(mask, 0, -cut_x)

final_x = shifted_x * anti_shifted_x


final_x

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.19911352, 1.        , 1.        , 1.        ,
        0.8       , 0.        , 0.        , 0.        ],
       [0.        , 0.19911352, 1.        , 1.        , 1.        ,
        0.8       , 0.        , 0.        , 0.        ],
       [0.        , 0.19911352, 1.        , 1.        , 1.        ,
        0.8       , 0.        , 0.        , 0.        ],
       [0.        , 0.19911352, 1.        , 1.        , 1.        ,
        0.8       , 0.        , 0.        , 0.        ],
       [0.        , 0.19911352, 1.        , 1.        , 1.        ,
        0.8       , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],


In [ ]:
shiftx = 0.2
erosion_x = np.sign(shiftx) * 1.4
cut_x = erosion_x - np.sign(erosion_x) * shiftx

shifted_x = fshift(mask, 0, shiftx)
anti_shifted_x = fshift(mask, 0, -cut_x)

final_x = shifted_x * anti_shifted_x


shifty = 0.2
erosion_y = np.sign(shifty) * 1.3
cut_y = erosion_y - np.sign(erosion_y) * shifty

shifted_y = fshift(mask, shifty, 0)
anti_shifted_y = fshift(mask, -cut_y, 0)

final_y = shifted_y * anti_shifted_y


final_x * final_y